<a href="https://colab.research.google.com/github/OscarPellicer/extra-attributions/blob/main/notebooks/extended/03_llm_interpretability.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Looking inside an LLM: logit lens, probes, sparse autoencoders and steering

In the slides we saw that **mechanistic interpretability** tries to explain a model in terms of its internal
representations. In this notebook we put some of those ideas into practice on **GPT-2 small** (124M parameters, 12 layers,
residual stream of dimension 768):

1. **Setup**: load GPT-2 with [TransformerLens](https://github.com/TransformerLensOrg/TransformerLens) and look at its next-token predictions.
2. **Logit lens**: read the residual stream of *intermediate* layers with the unembedding matrix.
3. **Linear probes**: test whether a concept (sentiment) is *linearly* decodable at each layer, with a control.
4. **Sparse autoencoders (SAEs)**: decompose the residual stream into (hopefully) monosemantic features with a
   pretrained SAE from [SAELens](https://github.com/jbloomAus/SAELens), and inspect them on [Neuronpedia](https://www.neuronpedia.org).
5. **Feature steering**: add an SAE feature direction to the residual stream and watch the generations change.
6. **Going further**: attribution graphs / circuit tracing.

⏱️ ~20-25 minutes. Everything runs on CPU, but a GPU (Colab T4, CUDA or Apple MPS) is faster.
The first run downloads GPT-2 (~500 MB) and the SAE weights (~300 MB) from Hugging Face.

**Exercises** only require **changing the values** of the parameters marked with `# change me` and re-running the cells.

## 0. Install & imports

In [ ]:
import sys

if 'google.colab' in sys.modules:
    # Colab: install the two interpretability libraries (torch, sklearn, matplotlib... are preinstalled)
    !pip install -q sae-lens "transformer-lens<4"
else:
    print("Running locally: make sure you installed the requirements, e.g.\n"
          "    pip install -r requirements.txt")

In [ ]:
import os, warnings
os.environ["TRANSFORMERLENS_ALLOW_MPS"] = "1"  # we check MPS correctness ourselves below
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

import random
from functools import partial

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display, HTML, IFrame

from transformer_lens import HookedTransformer
from sae_lens import SAE

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def set_seed(seed=0):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

set_seed(0)
torch.set_grad_enabled(False)  # we never need gradients in this notebook

# Device: cuda > mps > cpu
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print(f"Using {device=}")

## 1. Setup: GPT-2 small and its next-token predictions

`HookedTransformer` is a re-implementation of GPT-2 where **every intermediate activation has a named hook**
(e.g. `blocks.8.hook_resid_pre` = residual stream entering layer 8), which we can read (*cache*) or modify.

In [ ]:
model = HookedTransformer.from_pretrained("gpt2", device=device)
model.eval()

# Sanity check: some PyTorch versions give silently wrong results on Apple MPS -> compare against CPU, fall back if needed
if device == "mps":
    test = "The quick brown fox jumps over the lazy"
    try:
        logits_mps = model(test).float().cpu()
        model.to("cpu"); logits_cpu = model(test).float()
        ok = torch.allclose(logits_mps, logits_cpu, atol=1e-2)
    except Exception as e:
        print("MPS error:", e); ok = False
    if ok:
        model.to("mps")
    else:
        device = "cpu"
        print("⚠️ MPS results differ from CPU (or MPS failed): falling back to CPU")
print(f"Model on {device}: {model.cfg.n_layers} layers, d_model={model.cfg.d_model}, vocab={model.cfg.d_vocab}")

In [ ]:
PROMPT = "The Colosseum is located in the city of"  # change me
TOP_K = 10                                          # change me

def show_top_predictions(prompt, k=TOP_K):
    logits = model(prompt)                       # [batch=1, n_tokens, vocab]
    probs = logits[0, -1].softmax(-1)            # distribution over the NEXT token
    p, idx = probs.topk(k)
    print(f"Tokens: {model.to_str_tokens(prompt)}")
    for rank, (pi, ti) in enumerate(zip(p, idx)):
        print(f"  {rank:2d}. {model.to_string(ti)!r:15} {100 * pi.item():5.1f}%")

show_top_predictions(PROMPT)

### Exercise 1.1
Change `PROMPT` to `"The Eiffel Tower is located in the city of"`. Is GPT-2 small right? What about
`"The capital of France is"`?

*Hint:* look at the rank and probability of `' Paris'`. Remember that GPT-2 small is a tiny 2019 model.

<details><summary>Answer</summary>

For the Eiffel Tower prompt GPT-2 small predicts `' London'` first and `' Paris'` second (~7%): the "knowledge" is
there, but weak and noisy. For `"The capital of France is"` the top predictions are generic continuations (`' the'`,
`' a'`...), and `' Paris'` gets very low probability: the model does not interpret the sentence as a question to be
answered. Small LMs predict *plausible text*, not *facts*: a good reminder that we interpret what the model does,
not what we would like it to do.
</details>

## 2. Logit lens

The residual stream is a sum: $x_L = x_0 + \sum_{l<L} (\text{attn}_l + \text{mlp}_l)$. The **logit lens**
(nostalgebraist, 2020) applies the *final* LayerNorm and the unembedding $W_U$ to the residual stream after
each layer, as if the model stopped there: $\text{logits}_l = \text{LN}(x_l)\, W_U$.
This tells us *when* the prediction becomes linearly readable in the "output vocabulary basis".

In [ ]:
PROMPT = "The Colosseum is located in the city of"  # change me
TARGET = " Rome"                                    # change me (note the leading space!)

target_id = model.to_single_token(TARGET)
_, cache = model.run_with_cache(PROMPT)

# Residual stream at the last position before each layer + final output: [n_layers+1, d_model]
resid, labels = cache.accumulated_resid(apply_ln=True, pos_slice=-1, return_labels=True)
layer_logits = model.unembed(resid)[:, 0]          # [n_layers+1, vocab]
layer_probs = layer_logits.softmax(-1)

ranks = (layer_logits > layer_logits[:, [target_id]]).sum(-1).cpu().numpy()
probs = layer_probs[:, target_id].cpu().numpy()
top1 = [model.to_string(i) for i in layer_logits.argmax(-1)]

fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].plot(ranks + 1, "o-"); ax[0].set_yscale("log"); ax[0].invert_yaxis()
ax[0].set_title(f"Rank of {TARGET!r} (1 = top prediction)"); ax[0].set_xlabel("layer")
ax[1].plot(probs, "o-", color="C1"); ax[1].set_title(f"Probability of {TARGET!r}"); ax[1].set_xlabel("layer")
for a in ax:
    a.set_xticks(range(len(labels))); a.set_xticklabels(range(len(labels))); a.grid(alpha=.3)
plt.suptitle(f"Logit lens: {PROMPT!r}"); plt.tight_layout(); plt.show()

print("Top-1 token decoded at each layer (layer 12 = model output):")
print(pd.DataFrame({"top-1 token": top1, f"rank of {TARGET!r}": ranks + 1, "p": probs.round(3)}).T.to_string())

We can also apply the logit lens to **every position** at once: each cell shows the top-1 token decoded from the
residual stream of that token at that layer (i.e. what the model "would predict next" at that point).

In [ ]:
resid_all = cache.accumulated_resid(apply_ln=True)[:, 0]          # [n_layers+1, n_tokens, d_model]
top_tokens = model.unembed(resid_all).argmax(-1).cpu()            # [n_layers+1, n_tokens]
table = pd.DataFrame([[model.to_string(t) for t in row] for row in top_tokens],
                     columns=[f"{i}:{t}" for i, t in enumerate(model.to_str_tokens(PROMPT))])
table.index.name = "layer"
display(table.iloc[::2])   # every 2nd layer to keep it compact

### Exercise 2.1
With the default prompt (Colosseum → `' Rome'`), look at the probability plot. Where is the model most confident
about `' Rome'`? What happens in the last two layers? Compare with the top-10 list of Section 1.

<details><summary>Answer</summary>

`' Rome'` becomes the top-1 token at layer 9 and its logit-lens probability peaks at layer 10 (≈0.86), but at the
output it drops to ≈0.02, still top-1 but almost tied with tokens like `' P'`, `' L'`, `' T'`. The last layers
*reduce* confidence and spread probability over many alternatives (the start of other city names, e.g. *P*aris,
*L*ondon). Late layers of LMs are known to perform this kind of "hedging"/calibration. It also shows that the
logit lens of intermediate layers is not a calibrated prediction: we should compare ranks/trends rather than
absolute probabilities.
</details>

### Exercise 2.2
Set `PROMPT = "The Eiffel Tower is located in the city of"` and `TARGET = " Paris"`. At which layer does `' Paris'`
enter the top 10? Which token "wins" at the end, and from which layer?

*Hint:* read the table printed under the plot.

<details><summary>Answer</summary>

`' Paris'` is ranked > 1000 in the first layers, climbs to rank ≈11 at layer 9 and ends at rank 2.
`' London'` becomes the top-1 prediction from layer 10 onward. The logit lens shows that the "geographic city"
information is assembled in the middle-late layers (8-10), and the last layers sharpen (or, here, mis-sharpen)
the choice. Early layers decode to generic tokens like `' the'`: they are not yet in "output space".
</details>

### Exercise 2.3
Compare `PROMPT = "The Statue of Liberty is in New"`, `TARGET = " York"` with
`PROMPT = "Monday, Tuesday, Wednesday,"`, `TARGET = " Thursday"`. Which one is "solved" earlier? Why?

<details><summary>Answer</summary>

`' York'` after `' New'` is already the top-1 token from layer 1: it is essentially a **bigram** ("New" → "York")
that can be read almost directly from the current token's embedding. `' Thursday'` only becomes top-1 around
layer 10, because it requires combining information from *several previous tokens* (attention must move
information across positions, i.e. induction/sequence-continuation circuits). Caveat: the logit lens is only a
rough tool; early layers may not be in the same basis as the output (the *tuned lens* learns a per-layer
affine map to fix this).
</details>

## 3. Linear probes

A **linear probe** is a simple (linear) classifier trained on the hidden activations to predict a property.
If it works well at layer $l$, the property is *linearly decodable* there (linear representation hypothesis).
Below we build a tiny sentiment dataset with templates, collect the residual stream at the **last token**
of each sentence for every layer, and train a logistic regression per layer.

⚠️ Probe pitfalls: a good probe accuracy shows the information is *present*, not that the model *uses* it.
And a powerful probe can learn the task by itself. Hence we use a **control**: the same probe trained on a **control task**
(a random label assigned to each adjective), and we evaluate on **held-out words** never seen in training.

In [ ]:
POSITIVE = ["great", "wonderful", "excellent", "amazing", "fantastic", "lovely", "brilliant", "delightful",
            "superb", "pleasant", "enjoyable", "beautiful", "perfect", "charming", "fun", "awesome"]
NEGATIVE = ["terrible", "awful", "horrible", "dreadful", "boring", "disappointing", "bad", "poor",
            "miserable", "unpleasant", "annoying", "ugly", "disgusting", "mediocre", "painful", "useless"]
TEMPLATES = ["The movie was {}.", "I thought the food was {}.", "Honestly, the service was {}.",
             "My day was {}.", "The concert last night was {}.", "Overall, the book is {}."]
INCLUDE_NEGATION = False   # change me: True also adds "... was not {}." sentences with the label flipped

sentences, labels, words = [], [], []
for t in TEMPLATES:
    variants = [(t, False)] + ([(t.replace(" {}", " not {}"), True)] if INCLUDE_NEGATION else [])
    for tt, negated in variants:
        for w in POSITIVE: sentences.append(tt.format(w)); labels.append(int(not negated)); words.append(w)
        for w in NEGATIVE: sentences.append(tt.format(w)); labels.append(int(negated)); words.append(w)
labels, words = np.array(labels), np.array(words)
print(len(sentences), "sentences, e.g.:", sentences[0], sentences[16], sentences[-1])

In [ ]:
PROBE_POSITION = -1   # change me: -1 = final '.', -2 = last token of the adjective

layers = list(range(model.cfg.n_layers + 1))                # 0 = embeddings, l = after block l-1
hooks = [f"blocks.{l}.hook_resid_pre" for l in range(model.cfg.n_layers)] + \
        [f"blocks.{model.cfg.n_layers - 1}.hook_resid_post"]
acts = np.zeros((len(layers), len(sentences), model.cfg.d_model), dtype=np.float32)
for i, s in enumerate(sentences):
    _, c = model.run_with_cache(s, names_filter=lambda n: n in hooks)
    for l, h in enumerate(hooks):
        acts[l, i] = c[h][0, PROBE_POSITION].float().cpu().numpy()
print("Activations:", acts.shape, "(layers, sentences, d_model)")

In [ ]:
SPLIT_BY_WORD = True   # change me: True = test on held-out adjectives, False = random split of sentences
C = 0.1                # change me: inverse regularization strength of the logistic regression

rng = np.random.default_rng(0)
if SPLIT_BY_WORD:
    test_words = set(rng.choice(POSITIVE, 5, replace=False)) | set(rng.choice(NEGATIVE, 5, replace=False))
    test = np.isin(words, list(test_words))
else:
    test = rng.random(len(sentences)) < 0.3
train = ~test
# Control task (Hewitt & Liang, 2019): each adjective gets a RANDOM label, unrelated to its sentiment
all_words = POSITIVE + NEGATIVE
random_word_label = dict(zip(all_words, rng.permutation([0, 1] * (len(all_words) // 2))))
control_labels = np.array([random_word_label[w] for w in words])

def probe_acc(X, y):
    clf = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=2000))
    clf.fit(X[train], y[train])
    return clf.score(X[test], y[test])

acc_real = [probe_acc(acts[l], labels) for l in layers]
acc_ctrl = [probe_acc(acts[l], control_labels) for l in layers]

plt.figure(figsize=(7, 3.5))
plt.plot(layers, acc_real, "o-", label="sentiment labels")
plt.plot(layers, acc_ctrl, "s--", label="control: random label per word")
plt.axhline(0.5, color="gray", lw=1, ls=":", label="chance")
plt.xlabel("layer (0 = embeddings)"); plt.ylabel("test accuracy"); plt.ylim(0, 1.05); plt.xticks(layers)
plt.title(f"Linear probe, position {PROBE_POSITION}, split_by_word={SPLIT_BY_WORD}")
plt.legend(); plt.grid(alpha=.3); plt.show()
print("real:", np.round(acc_real, 2)); print("ctrl:", np.round(acc_ctrl, 2))

### Exercise 3.1
Look at the plot with the default settings (`PROBE_POSITION = -1`). Why is accuracy at chance at layer 0?
At which layer does sentiment become linearly decodable at the final `.` token?
What does the control task tell us?

<details><summary>Answer</summary>

At layer 0 the residual stream at position -1 contains only the embedding of `'.'` (+ its position), which is
(almost) identical for all sentences: there is nothing to decode, so the probe is at chance. Already after layer 1,
attention has moved the adjective's sentiment to the final token and the probe reaches 100% on *held-out adjectives*.
The control task stays at or below chance (≈0.2-0.5): random word labels cannot be generalized to unseen words.
The gap between the two curves (*selectivity*) tells us the high accuracy reflects structure in the
representation, not just the capacity of the probe. (This toy task is easy; real concepts are rarely this clean.)
</details>

### Exercise 3.2
Set `PROBE_POSITION = -2` (the adjective token itself) and re-run the two probe cells. Then also set
`SPLIT_BY_WORD = False`. What happens at layer 0? What happens to the control?

*Hint:* with a random split, the *same adjective* appears both in training and test sentences.

<details><summary>Answer</summary>

At the adjective token, sentiment is decodable with 100% accuracy already from the **token embedding** (layer 0):
word embeddings encode sentiment, no "processing" needed. With `SPLIT_BY_WORD = False` the test sentences contain
adjectives seen during training, and now the **control task also reaches ~100%** in the early layers: the probe
simply **memorizes word identity → label**, even though those labels are meaningless. A probe with high accuracy
on a leaky split proves nothing. This is why proper (held-out) splits and control tasks (Hewitt & Liang, 2019)
are essential.
</details>

### Exercise 3.3
Set `SPLIT_BY_WORD = True`, `INCLUDE_NEGATION = True` (this adds sentences like *"The movie was not great."* with
the label flipped) and re-run the three probe cells, first with `PROBE_POSITION = -1`, then with `PROBE_POSITION = -2`.
How does the accuracy curve change compared to the version without negation?

<details><summary>Answer</summary>

Now the probe must decode the sentiment of the *whole phrase*, not of the adjective alone. At the final `.` (position -1)
accuracy grows gradually (≈0.73 at layer 1, ≈0.95 at layer 3, 100% from layer 5). At the adjective token (position -2)
layer 0 drops to chance (the embedding of *great* is the same in "was great" and "was not great"), and accuracy only
reaches ~100% at layer 4, once attention has incorporated the preceding "not". Probing across layers lets us see
**where** the model composes meaning from context.
</details>

## 4. Sparse autoencoders (SAEs)

Neurons are polysemantic (superposition). An **SAE** re-expresses an activation $x \in \mathbb{R}^{768}$ as a sparse,
non-negative combination of many more **features** (here 24,576):

$$f(x) = \text{ReLU}(W_{enc}(x - b_{dec}) + b_{enc}), \qquad \hat x = W_{dec}\, f(x) + b_{dec}$$

trained to reconstruct $x$ with an L1 sparsity penalty. Each row of $W_{dec}$ is a **feature direction**.
We use Joseph Bloom's open residual-stream SAEs for GPT-2 small (`gpt2-small-res-jb`), at the input of layer 8.

In [ ]:
SAE_RELEASE, SAE_LAYER = "gpt2-small-res-jb", 8
HOOK = f"blocks.{SAE_LAYER}.hook_resid_pre"

sae = SAE.from_pretrained(SAE_RELEASE, HOOK, device=device)
if isinstance(sae, tuple):  # sae-lens < 6 returned (sae, cfg_dict, sparsity)
    sae = sae[0]
sae.eval()
print(f"SAE: d_in={sae.cfg.d_in}, d_sae={sae.cfg.d_sae}  ->  {sae.cfg.d_sae / sae.cfg.d_in:.0f}x overcomplete")

In [ ]:
TEXT = ("After the ceremony in the old chapel, the bride and groom walked down the aisle while the guests "
        "threw rice. The reception was held in a garden and everyone danced until midnight.")  # change me

tokens = model.to_tokens(TEXT)                      # prepends <|endoftext|> (BOS), as the SAE was trained with it
str_tokens = model.to_str_tokens(TEXT)
_, cache = model.run_with_cache(tokens, names_filter=HOOK)
x = cache[HOOK][0]                                  # [n_tokens, 768]
feats = sae.encode(x)                               # [n_tokens, 24576]
x_hat = sae.decode(feats)

l0 = (feats[1:] > 0).sum(-1).float()                # skip BOS (huge norm, atypical)
fvu = ((x[1:] - x_hat[1:]) ** 2).sum() / ((x[1:] - x[1:].mean(0)) ** 2).sum()
print(f"L0 (active features per token): mean {l0.mean():.1f} of {sae.cfg.d_sae}  ({100 * l0.mean() / sae.cfg.d_sae:.2f}%)")
print(f"Variance explained by the reconstruction: {100 * (1 - fvu):.1f}%")

# Does the model still work if we replace the residual stream by the SAE reconstruction? (CE loss)
def splice_sae(act, hook):    return sae.decode(sae.encode(act)).to(act.dtype)
def zero_ablate(act, hook):   return torch.zeros_like(act)
ce_clean = model(tokens, return_type="loss").item()
ce_sae   = model.run_with_hooks(tokens, return_type="loss", fwd_hooks=[(HOOK, splice_sae)]).item()
ce_zero  = model.run_with_hooks(tokens, return_type="loss", fwd_hooks=[(HOOK, zero_ablate)]).item()
print(f"Next-token CE loss: clean {ce_clean:.2f} | SAE spliced in {ce_sae:.2f} | zero-ablated {ce_zero:.2f}")
print(f"-> CE loss recovered: {100 * (ce_zero - ce_sae) / (ce_zero - ce_clean):.1f}%")

In [ ]:
N_TOP_FEATURES = 3   # change me

vals, idx = feats.topk(N_TOP_FEATURES, dim=-1)
rows = []
for i, tok in enumerate(str_tokens):
    rows.append({"token": repr(tok), **{f"feat #{k+1}": f"{idx[i, k].item()} ({vals[i, k].item():.1f})"
                                       for k in range(N_TOP_FEATURES)}})
display(pd.DataFrame(rows).iloc[1:])   # skip BOS

Each feature has a page on **Neuronpedia** with its top-activating examples from a large corpus, an auto-generated
(LLM-written) explanation, and the tokens it promotes in the output. Let us look at feature **5897**: we picked it
because it is the top feature on `' bride'` and `' groom'` in the table above.
Below we compute its activation over a small set of texts.

In [ ]:
FEATURE = 5897   # change me (try other indices from the table above)

TEXTS = [
    "They got married last June; the wedding ceremony was small and the bride wore her mother's gown.",
    "We sent the invitations for our anniversary party and booked a band for the reception.",
    "The groom's best man gave a funny speech and then they cut the cake.",
    "The stock market fell sharply on Monday after the central bank raised interest rates.",
    "To make the sauce, fry the garlic in olive oil and add the chopped tomatoes.",
    "The astronauts docked with the space station after a six-hour flight.",
    "The striker scored twice in the second half and the team won the championship.",
    "Python lists are mutable, while tuples cannot be changed after creation.",
    "Santa brought presents and we sang carols around the Christmas tree.",
    "The funeral procession moved slowly through the village towards the church.",
]

def feature_acts(text, feature):
    toks = model.to_tokens(text)
    _, c = model.run_with_cache(toks, names_filter=HOOK)
    return model.to_str_tokens(text)[1:], sae.encode(c[HOOK][0, 1:])[:, feature].cpu().numpy()

def html_highlight(str_toks, acts, vmax):
    spans = "".join(f'<span style="background: rgba(255,120,0,{min(a / vmax, 1):.2f}); color: black" '
                    f'title="{a:.2f}">{t.replace("<", "&lt;")}</span>' for t, a in zip(str_toks, acts))
    return f'<div style="font-family: monospace; line-height: 1.8; background: white; padding: 2px">{spans}</div>'

results = [feature_acts(t, FEATURE) for t in TEXTS]
vmax = max(max(a.max() for _, a in results), 1e-6)
display(HTML(f"<b>Feature {FEATURE}</b> (orange = active, hover for value)" +
             "".join(html_highlight(s, a, vmax) for s, a in results)))

# Top-activating tokens across these texts
all_acts = [(a, s[j], i) for i, (s, acts) in enumerate(results) for j, a in enumerate(acts)]
print("Top activating tokens:")
for a, tok, i in sorted(all_acts, reverse=True)[:10]:
    print(f"  {a:6.2f}  {tok!r:15} (text {i})")

url = f"https://www.neuronpedia.org/gpt2-small/{SAE_LAYER}-res-jb/{FEATURE}"
print(f"\n👉 Inspect this feature on Neuronpedia: {url}")

In [ ]:
# Optional: embed the Neuronpedia page (needs internet; if it does not render, just open the URL above)
SHOW_IFRAME = True   # change me
if SHOW_IFRAME:
    display(IFrame(f"{url}?embed=true&embedexplanation=true&embedplots=true&embedtest=true", width=900, height=500))

### Exercise 4.1
Is feature 5897 *monosemantic* on these texts? Does it fire on `funeral` / `procession` or `church` (related
ceremonies, but not weddings)? Compare with its Neuronpedia page (explanation and top examples).

<details><summary>Answer</summary>

It fires strongly on wedding words (`wedding` ≈ 50, `bride` ≈ 30, `groom`, `ceremony`, `married`) and is silent on
unrelated texts (finance, cooking, space, code, Christmas). Neuronpedia's explanation is "words related to weddings".
However, it also fires on `funeral` (≈ 10): a related but different ceremony. Features often have fuzzy borders,
and an explanation (especially an auto-generated one) is a *hypothesis* that should be tested with
counterexamples like these, not a guaranteed definition.
</details>

### Exercise 4.2
Change `TEXT` to a sentence about a completely different topic (e.g. football or cooking) and re-run.
Pick one of the top features and set `FEATURE` to it. Does Neuronpedia's explanation match the tokens where it fires?
Also: what is the mean L0? What fraction of the 24,576 features is active per token?

<details><summary>Answer</summary>

Typically only ~65 out of 24,576 features (≈0.26%) are active per token: the code is very sparse, yet the
reconstruction explains ≈83% of the variance, and splicing it into the model recovers ≈98% of the CE loss
(clean ≈3.5 vs spliced ≈3.7, while zero-ablation gives ≈11). The remaining error is the SAE's "dark matter". Topic-specific features (e.g. 14149 "ingredients in recipes" for cooking)
usually have clear explanations; other top features are generic (e.g. punctuation, position, grammar) and
harder to name. Some features are "dense" and fire almost everywhere.
</details>

## 5. Feature steering

If a feature direction $d_f$ (row of $W_{dec}$) causally represents a concept, **adding** $\alpha\, d_f$
to the residual stream should make the model "think" about it (cf. *Golden Gate Claude*). We add it at layer 8,
at every position, with a forward hook, and sample generations with a fixed seed.

(The residual stream norm at layer 8 is ≈ 100, and decoder directions have unit norm, so $\alpha$ of a few
tens is a sizeable but not destructive push.)

In [ ]:
STEER_FEATURE = 5897                 # change me (5897 weddings; try 14149 recipes, 3313 space missions, 8526 Christmas)
STEER_COEF = 60                      # change me (try 0, 20, 60, 150, 400, -60)
STEER_PROMPT = "I think that"        # change me
N_SAMPLES, MAX_NEW_TOKENS, SEED = 3, 30, 0

steer_dir = sae.W_dec[STEER_FEATURE].detach().clone()

def steering_hook(act, hook, coef):
    return act + coef * steer_dir.to(act.dtype)

def generate(prompt, coef, seed=SEED):
    set_seed(seed)
    with model.hooks(fwd_hooks=[(HOOK, partial(steering_hook, coef=coef))]):
        return model.generate(prompt, max_new_tokens=MAX_NEW_TOKENS, temperature=0.7, top_p=0.9,
                              freq_penalty=1.0, verbose=False)

print(f"Feature {STEER_FEATURE}: https://www.neuronpedia.org/gpt2-small/{SAE_LAYER}-res-jb/{STEER_FEATURE}\n")
for s in range(N_SAMPLES):
    print(f"--- sample {s} ---")
    print("  normal :", repr(generate(STEER_PROMPT, 0, seed=SEED + s)))
    print("  steered:", repr(generate(STEER_PROMPT, STEER_COEF, seed=SEED + s)))

Note: the hand-picked features and coefficients were found empirically (a few tries) to give a visible effect.
Generations are sampled (fixed seed), so results may differ slightly across devices and library versions.

### Exercise 5.1
Sweep `STEER_COEF` over `0, 20, 60, 150, 400`. Describe what happens. Is there a "sweet spot"?

<details><summary>Answer</summary>

At 0 the generations are identical to the unsteered ones (sanity check). At 20 the effect is barely visible;
around 60 the text drifts towards the concept ("party planner", "wedding reception", "groomed") while staying
grammatical; at 150-400 the steering vector dominates the residual stream (norm ≈ 100) and the output degenerates
into lists of wedding words ("reception reception ceremony chapel vows..."). The sweet spot is where the concept
appears without destroying fluency. Steering is a **causal**
intervention: unlike probes, it shows that the direction is *used* by the downstream layers.
</details>

### Exercise 5.2
Try `STEER_FEATURE = 14149` (recipes), `3313` (space missions) or `8526` (Christmas). Do all features steer equally well
with the same coefficient? Check their Neuronpedia pages.

<details><summary>Answer</summary>

With the same coefficient, 14149 (recipes: "tomato sauce", "ranch dressing", "chicken") and 3313 (space missions:
"flyby", "moon landing", "spacecraft") produce even clearer topic shifts than the wedding feature; 8526 also works
("Christmas Eve", "holiday season"). Features differ in how strongly they control
the output: some are more "input-side" (they detect a concept) and others more "output-side" (they promote
specific tokens), and the useful coefficient depends on the typical activation of the feature.
</details>

### Exercise 5.3
Set `STEER_PROMPT = "The bride and groom"` and `STEER_COEF = -60` (negative steering, i.e. *removing* the wedding
direction). Compare with `STEER_COEF = 0`.

<details><summary>Answer</summary>

Without steering, continuations talk about the wedding ("photographed at the wedding", "wearing dresses").
With −60 the wedding context disappears and the text drifts to unrelated news-like topics (families, intelligence
agencies, crimes...), even though the prompt still says "bride and groom". Negative steering can suppress a
concept, but the result is not "the same text minus weddings": it pushes the model into an unusual region of
activation space and the generations become odd. In other cases removing one direction is not enough, as
concepts are often represented redundantly by several correlated features.
</details>

## 6. Going further: attribution graphs and circuit tracing

SAE features tell us *what* is represented at one layer; **circuits** tell us *how* features at different layers
interact to produce an output. In 2025 Anthropic published
[*Circuit Tracing*](https://transformer-circuits.pub/2025/attribution-graphs/methods.html) and
[*On the Biology of a Large Language Model*](https://transformer-circuits.pub/2025/attribution-graphs/biology.html):
they replace the MLPs by **cross-layer transcoders** and compute **attribution graphs**, i.e. the
linear (per-prompt) effects between interpretable features, from input tokens to the output logit
(e.g. "Dallas" → "Texas" → "capital" → "Austin" for multi-hop reasoning).

To explore on your own:
- **Neuronpedia's graph UI**: [neuronpedia.org/gemma-2-2b/graph](https://www.neuronpedia.org/gemma-2-2b/graph):
  generate and explore attribution graphs for your own prompts in the browser, no code required.
- **`circuit-tracer`** library ([github.com/safety-research/circuit-tracer](https://github.com/safety-research/circuit-tracer)):
  compute attribution graphs for Gemma-2-2B / Llama-3.2-1B with pretrained transcoders, and intervene on features.
- **SAELens** and **Neuronpedia** host thousands of pretrained SAEs for many open models (Gemma Scope, Llama Scope...).

Takeaways from this notebook:
- The **logit lens** reveals *when* a prediction forms, but early layers are not in "output space".
- **Probes** reveal what is *decodable*, which needs controls and careful splits, and is not evidence of *use*.
- **SAEs** give a sparse, often interpretable basis, with imperfect reconstruction (the "dark matter").
- **Steering** is a causal test: if adding a direction changes behaviour as expected, the model uses it.